# RQ3 — Robustezza alla perdita del pivot strutturale

**Progetto:** Reti dei passaggi delle semifinaliste del Mondiale 2022
**Insegnamento:** Analisi e Visualizzazione delle Reti Complesse · Final project
**Lezioni di riferimento:** NS06 (centralità), NS07 (robustezza), NS08 (modelli nulli)
**Lettura:** Albert, Jeong & Barabási (2000), *Error and attack tolerance of complex networks*

---

## La domanda, in una sola frase

> *Se rimuovessimo il giocatore più centrale di ciascuna squadra, di quanto si romperebbe davvero la sua rete dei passaggi — e quel danno sarebbe peggiore di quello prodotto dalla perdita di un giocatore di movimento qualsiasi?*

La domanda ha una lettura strutturale e una tattica. **Strutturalmente**, è l'esperimento di attacco mirato della Lezione 7 applicato a una piccola rete pesata diretta ad alta densità. **Tatticamente**, chiede quanto ciascuna delle quattro semifinaliste del 2022 dipendesse da un singolo giocatore per tenere insieme la propria struttura di passaggi.

Il notebook è organizzato in cinque movimenti:

| § | Sezione | Cosa facciamo |
|---|---|---|
| 1 | **Impostazione strutturale** | Carichiamo le quattro reti aggregate; registriamo il *rapporto di dominanza del pivot* dalla RQ1. |
| 2 | **Metriche di danno** | Adattiamo $S(q)$ della NS07 a due metriche adatte a reti dense pesate dirette. |
| 3 | **Attacco a singolo nodo** | Confrontiamo rimozione mirata vs casuale (n = 200) e trasformiamo il divario in z-score. |
| 4 | **Progressione dell'attacco** | Rimozione cumulativa dei top-3, ricalcolando la betweenness a ogni passo. |
| 5 | **Il predittore** | Verifichiamo se il rapporto di dominanza della RQ1 predice il ranking di fragilità della RQ3. |

Ogni sezione si chiude con una riga di sintesi — il "cosa abbiamo imparato" — in modo che il report finale si possa praticamente scrivere a partire dalle chiusure di sezione.


In [ ]:
# -----------------------------------------------------------------------------
# Setup — typography, palette, figure-size scale, helpers.
# Mirrors the conventions of notebooks 04 (game-of-thrones) and 05 (spatial).
# -----------------------------------------------------------------------------
from netsci_utils import *
import pickle
import pandas as pd
from textwrap import fill
from scipy.stats import pearsonr, spearmanr
from matplotlib.patches import Rectangle, Circle, Arc, FancyBboxPatch

set_seeds()

# High-resolution inline rendering, as in notebook 05.
try:
    from IPython import get_ipython
    get_ipython().run_line_magic("config", "InlineBackend.figure_format = 'retina'")
except Exception:
    pass
plt.rcParams["figure.dpi"]     = 150
plt.rcParams["savefig.dpi"]    = 220
plt.rcParams["savefig.bbox"]   = "tight"
plt.rcParams["axes.grid"]      = False
plt.rcParams["font.size"]      = 11
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.edgecolor"]    = "#475569"
plt.rcParams["axes.labelcolor"]   = "#0F172A"
plt.rcParams["xtick.color"]       = "#475569"
plt.rcParams["ytick.color"]       = "#475569"

# Typography scale (mirrors TEXT from notebooks 04 and 05).
TEXT = {
    "fig_title":   16,
    "fig_subtitle": 11,
    "panel_title": 12,
    "annotation":   9.5,
    "direct_label": 9.5,
    "kpi_number":  20,
    "kpi_label":    9.2,
    "micro":        8.8,
}

# Figure-size scale.
FIG = {
    "focus":    (8.0, 6.0),
    "standard": (10.0, 6.0),
    "wide":     (12.5, 6.0),
    "flagship": (13.5, 12.5),
    "grid":     (12.5, 11.0),
}

# Colour system. INK = primary text, INK_SOFT = secondary text, MUTED = separators.
INK      = "#0F172A"
INK_SOFT = "#475569"
MUTED    = "#E2E8F0"
BG       = "#FAFBFC"

# Course-aware palette, expanded with team colours for the four semifinalists.
DV_PALETTE = {
    "blue":   "#4C72B0",
    "orange": "#DD8452",
    "green":  "#55A868",
    "red":    "#C44E52",
    "purple": "#8172B2",
    "gray":   "#7F8589",
}
ACCENT   = DV_PALETTE["orange"]
HIGHLIGHT = DV_PALETTE["red"]
PITCH_BG = "#1e3a2d"          # dark green for pitch backgrounds
PITCH_LN = "#e6efe6"          # pitch white lines

TEAM_COLORS = {
    "Argentina": "#75AADB",   # light blue (national)
    "France":    "#002395",   # dark blue
    "Croatia":   "#C44E52",   # red
    "Morocco":   "#006233",   # green
}

LINE_COLORS = {
    "GK":  "#FCBF49",
    "DEF": "#3A86FF",
    "MID": "#E63946",
    "ATT": "#06D6A0",
}


# -----------------------------------------------------------------------------
# Layout helpers — match the prof's style_panel / reserve_header pattern.
# -----------------------------------------------------------------------------
def style_panel(ax, title=None, subtitle=None, *, title_color=INK):
    """Two-tier header above an axis (panel-title + smaller subtitle)."""
    if title:
        ax.set_title(title, loc="left", pad=14 if subtitle else 8,
                     fontsize=TEXT["panel_title"], fontweight="semibold",
                     color=title_color)
    if subtitle:
        # Place subtitle just under the title, in INK_SOFT.
        ax.text(0.0, 1.02, subtitle, transform=ax.transAxes,
                ha="left", va="bottom",
                fontsize=TEXT["annotation"], color=INK_SOFT,
                fontstyle="italic")
    return ax


def text_below_axes(ax, text, *, y=-0.16, mono=True, box=True):
    """Outside-axes text block (mono-spaced stats line below the panel)."""
    bbox = (dict(boxstyle="round,pad=0.30", fc="white", ec=MUTED, lw=0.8,
                 alpha=0.97) if box else None)
    ax.text(0.5, y, text, transform=ax.transAxes, ha="center", va="top",
            fontsize=TEXT["annotation"],
            family="monospace" if mono else None,
            bbox=bbox, clip_on=False, zorder=20)


def add_kpi_strip(ax, kpis):
    """KPI band: tall number above smaller label. `kpis` is a list of (number, label)."""
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_axis_off()
    n = len(kpis)
    for i, (num, lab) in enumerate(kpis):
        x = (i + 0.5) / n
        ax.text(x, 0.62, num, ha="center", va="center",
                fontsize=TEXT["kpi_number"], fontweight="semibold", color=INK)
        ax.text(x, 0.22, lab, ha="center", va="center",
                fontsize=TEXT["kpi_label"], color=INK_SOFT)
    ax.plot([0.02, 0.98], [-0.02, -0.02], color=MUTED, lw=0.9, clip_on=False)


# -----------------------------------------------------------------------------
# Pitch drawing for the spatial mini-panels (StatsBomb 120x80 frame).
# -----------------------------------------------------------------------------
def draw_pitch(ax, color=PITCH_BG, line=PITCH_LN, lw=1.0):
    ax.add_patch(Rectangle((0, 0), 120, 80, fc=color, ec=line, lw=lw + 0.5))
    ax.plot([60, 60], [0, 80], color=line, lw=lw)
    ax.add_patch(Circle((60, 40), 9.15, fill=False, ec=line, lw=lw))
    ax.plot(60, 40, "o", color=line, ms=2)
    ax.add_patch(Rectangle((0, 18), 18, 44, fill=False, ec=line, lw=lw))
    ax.add_patch(Rectangle((102, 18), 18, 44, fill=False, ec=line, lw=lw))
    ax.add_patch(Rectangle((0, 30), 6, 20, fill=False, ec=line, lw=lw))
    ax.add_patch(Rectangle((114, 30), 6, 20, fill=False, ec=line, lw=lw))
    ax.plot(12, 40, "o", color=line, ms=2); ax.plot(108, 40, "o", color=line, ms=2)
    ax.add_patch(Arc((12, 40), 18.3, 18.3, angle=0, theta1=-53, theta2=53, color=line, lw=lw))
    ax.add_patch(Arc((108, 40), 18.3, 18.3, angle=0, theta1=127, theta2=233, color=line, lw=lw))
    # adjustable="box" preserves data limits and squashes the axes to fit,
    # which keeps the pitch in its true 120:80 ratio without "Ignoring x limits".
    ax.set_xlim(-2, 122); ax.set_ylim(-2, 82)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)


# -----------------------------------------------------------------------------
# Data + analysis helpers (logic only — figures live in separate cells).
# -----------------------------------------------------------------------------
DATA_DIR = "./networks"
TEAMS = ["Argentina", "France", "Croatia", "Morocco"]
N_RANDOM = 200
PIVOT_METRIC = "weighted betweenness"

ALIASES = {
    "Lionel Andrés Messi Cuccittini": "Messi",
    "Rodrigo Javier De Paul":          "De Paul",
    "Nicolás Hernán Otamendi":         "Otamendi",
    "Enzo Fernandez":                  "Fernandez",
    "Kylian Mbappé Lottin":            "Mbappé",
    "Aurélien Djani Tchouaméni":       "Tchouaméni",
    "Antoine Griezmann":               "Griezmann",
    "Randal Kolo Muani":               "Kolo Muani",
    "Achraf Hakimi Mouh":              "Hakimi",
    "Yahia Attiyat allah":             "Attiyat-Allah",
    "Luka Modrić":                     "Modrić",
    "Mateo Kovačić":                   "Kovačić",
    "Marcelo Brozović":                "Brozović",
    "Joško Gvardiol":                  "Gvardiol",
    "Sofyan Amrabat":                  "Amrabat",
}
def short(n):
    return ALIASES.get(n, n.split()[-1])


def load_wc2022_semifinalists():
    return {t: pickle.load(open(f"{DATA_DIR}/{t}.gpickle", "rb")) for t in TEAMS}


def add_distance(G):
    H = G.copy()
    for u, v, d in H.edges(data=True):
        d["distance"] = 1.0 / max(d["weight"], 1e-9)
    return H


def avg_path_length(G):
    if G.number_of_nodes() < 2: return float("nan")
    largest = max(nx.weakly_connected_components(G), key=len)
    H = add_distance(G.subgraph(largest).copy())
    lengths = dict(nx.all_pairs_dijkstra_path_length(H, weight="distance"))
    values = [d for u, t in lengths.items() for v, d in t.items() if u != v]
    return float(np.mean(values)) if values else float("nan")


def efficiency(G):
    if G.number_of_nodes() < 2: return float("nan")
    H = add_distance(G)
    lengths = dict(nx.all_pairs_dijkstra_path_length(H, weight="distance"))
    values = [1.0 / d for u, t in lengths.items() for v, d in t.items()
              if u != v and d > 0]
    return float(np.mean(values)) if values else float("nan")


def weighted_betweenness(G):
    return nx.betweenness_centrality(add_distance(G), weight="distance",
                                     normalized=True)


def relative_damage(pre, post, metric_name):
    if np.isnan(pre) or np.isnan(post) or pre <= 0: return float("nan")
    if metric_name == "avg_path_length":
        return (post - pre) / pre
    return (pre - post) / pre


---
## 1. Impostazione strutturale — quattro squadre, un numero per squadra

Ciascuna delle quattro semifinaliste produce un grafo diretto pesato con 20–23 nodi (giocatori che hanno disputato almeno 30 minuti per la nazionale) e 281–314 archi attivi (coppie di passaggio completate, sommate lungo tutto il torneo).

Un singolo numero riassume la §1 per quello che verrà dopo: il **rapporto top-1 / top-2 della weighted betweenness**. È il punteggio di dominanza strutturale che arriva dalla RQ1 — quanto il pivot della squadra svetta sopra il secondo giocatore più centrale. Un rapporto pari a 1 significherebbe che i primi due sono intercambiabili; un rapporto di 3 significa che il pivot è tre volte più centrale di chiunque altro.


In [ ]:
graphs = load_wc2022_semifinalists()

# Build the §1 summary table and the dominance score per team.
rows = []
for team in TEAMS:
    G = graphs[team]
    btw = weighted_betweenness(G)
    sorted_btw = sorted(btw.values(), reverse=True)
    top1_top2 = sorted_btw[0] / sorted_btw[1] if sorted_btw[1] > 0 else float("inf")
    pivot = max(btw, key=btw.get)
    rows.append({
        "team": team,
        "n":    G.number_of_nodes(),
        "edges":    G.number_of_edges(),
        "total passes":  sum(d["weight"] for _, _, d in G.edges(data=True)),
        "density":      nx.density(G),
        "reciprocity":  nx.reciprocity(G),
        "pivot":        short(pivot),
        "top1/top2":    top1_top2,
    })
setup_df = pd.DataFrame(rows)
print(setup_df.round(2).to_string(index=False))


In [ ]:
# A visual summary of the four-team setup: a KPI strip across the top,
# followed by per-team mini-cards with the pivot identified.
fig = plt.figure(figsize=FIG["wide"], facecolor=BG)
gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[1.0, 2.6],
                      left=0.05, right=0.97, top=0.93, bottom=0.06,
                      hspace=0.35)

# --- Title and subtitle directly on the figure
fig.text(0.05, 0.985, "Quattro squadre, quattro reti dei passaggi — una fotografia strutturale",
         ha="left", va="top", fontsize=TEXT["fig_title"],
         fontweight="semibold", color=INK)
fig.text(0.05, 0.955,
         "Reti aggregate per squadra-torneo delle semifinaliste del Mondiale 2022 FIFA. "
         "Rapporto top-1 / top-2 = il divario fra il giocatore più centrale e il secondo più centrale.",
         ha="left", va="top", fontsize=TEXT["fig_subtitle"], color=INK_SOFT)

# --- KPI strip: aggregate totals across the four teams
ax_kpi = fig.add_subplot(gs[0])
total_passes = int(setup_df["total passes"].sum())
total_players = int(setup_df["n"].sum())
mean_density = setup_df["density"].mean()
mean_reciprocity = setup_df["reciprocity"].mean()
add_kpi_strip(ax_kpi, [
    (f"{total_passes:,}",         "passaggi completati aggregati"),
    (f"{total_players}",          "giocatori con ≥ 30′ giocati"),
    (f"{mean_density:.2f}",       "densità media delle reti"),
    (f"{mean_reciprocity:.2f}",   "reciprocità media (passaggio e ritorno)"),
])

# --- Per-team mini-cards
ax_cards = fig.add_subplot(gs[1]); ax_cards.set_axis_off()
ax_cards.set_xlim(0, 1); ax_cards.set_ylim(0, 1)
for i, row in setup_df.iterrows():
    x0 = 0.025 + i * 0.245; x1 = x0 + 0.22
    team = row["team"]; color = TEAM_COLORS[team]
    # Left coloured stripe
    ax_cards.add_patch(Rectangle((x0, 0.10), 0.012, 0.80,
                                 fc=color, ec="none", clip_on=False))
    # Card body
    ax_cards.add_patch(FancyBboxPatch((x0 + 0.015, 0.10), 0.205, 0.80,
        boxstyle="round,pad=0.005,rounding_size=0.012",
        fc="white", ec=MUTED, lw=0.8, clip_on=False))
    # Team name
    team_it = {"Argentina":"Argentina","France":"Francia","Croatia":"Croazia","Morocco":"Marocco"}[team]
    ax_cards.text(x0 + 0.028, 0.82, team_it, ha="left", va="center",
                  fontsize=13, fontweight="semibold", color=INK)
    # Pivot
    ax_cards.text(x0 + 0.028, 0.72, f"pivot · {row['pivot']}", ha="left",
                  va="center", fontsize=10.5, color=INK_SOFT, fontstyle="italic")
    # Top1/top2 ratio - the highlighted number
    ax_cards.text(x0 + 0.028, 0.50, f"{row['top1/top2']:.2f}", ha="left",
                  va="center", fontsize=22, fontweight="semibold", color=color)
    ax_cards.text(x0 + 0.028, 0.36, "rapporto top-1 / top-2 della betweenness",
                  ha="left", va="center", fontsize=8.8, color=INK_SOFT)
    # Three smaller stats
    ax_cards.text(x0 + 0.028, 0.22,
                  f"{row['n']} giocatori · {row['edges']} archi · {int(row['total passes']):,} passaggi",
                  ha="left", va="center", fontsize=8.5, color=INK_SOFT)
    ax_cards.text(x0 + 0.028, 0.15,
                  f"densità {row['density']:.2f}   ·   reciprocità {row['reciprocity']:.2f}",
                  ha="left", va="center", fontsize=8.5, color=INK_SOFT,
                  family="monospace")
plt.show()


**Cosa abbiamo imparato.**
- Tutte e quattro le reti sono **dense** (0.57–0.75) e **fortemente reciproche** (0.81–0.90): grafi bidirezionali quasi completi.
- Il **Marocco** è l'outlier strutturale: densità più bassa, reciprocità più bassa, conteggio passaggi più basso. Coerente con uno stile meno basato sul possesso.
- Il **rapporto top-1 / top-2** copre un ordine di grandezza fra le quattro squadre: da **1.07** (Marocco — Amrabat appena davanti a Hakimi) a **3.20** (Francia — Tchouaméni un gigante sopra tutti gli altri).
- *Ipotesi per la §5:* questo rapporto dovrebbe predire quanto ogni squadra soffre quando le si toglie il pivot.


---
## 2. Metriche di danno — adattare $S(q)$ alle reti dense pesate

La variabile di robustezza della NS07

$$ S(q) = \frac{\text{dimensione della LCC dopo aver rimosso una frazione } q}{N} $$

non si trasferisce direttamente a grafi densi pesati e diretti: con densità di 0.57–0.75, **rimuovere un singolo nodo non frammenta mai la LCC** (lo abbiamo verificato su oltre 800 rimozioni di prova — vedi §7 della proposta). Servono metriche sensibili alla *degradazione strutturale che si verifica prima della frammentazione*.

Ne usiamo due:

- **Lunghezza media dei cammini** sulla LCC, con `1/peso` come distanza, in modo che passaggi frequenti corrispondano a geodetiche corte.
- **Efficienza** $E = \frac{1}{n(n-1)}\sum_{i \neq j} \frac{1}{d_{ij}}$ — la media della distanza reciproca su tutte le coppie dirette raggiungibili (Latora & Marchiori 2001). Valori più bassi significano un flusso di informazione/palla più lento.

Entrambe le metriche saranno riportate come **danno relativo**, con il segno scelto in modo che *valori positivi indichino sempre "peggio"*.


In [ ]:
baseline_rows = []
for team in TEAMS:
    G = graphs[team]
    baseline_rows.append({
        "team":             team,
        "avg path length":  avg_path_length(G),
        "efficiency":       efficiency(G),
    })
baseline_df = pd.DataFrame(baseline_rows).set_index("team")
print(baseline_df.round(3).to_string())


**Cosa abbiamo imparato.**
- L'**Argentina** ha i cammini più corti e l'efficienza più alta: una rete di possesso stretta e ben tessuta.
- Il **Marocco** ha i cammini più lunghi e l'efficienza più bassa — in media servono più passaggi per attraversare la rete, ancora coerente con uno stile meno basato sul possesso.
- Questi quattro numeri (× 2 metriche) sono i *valori di riferimento* contro cui misureremo, nella §3, il danno causato dalla rimozione di un nodo.


---
## 3. Attacco a singolo nodo — mirato vs casuale

Per ciascuna squadra ripetiamo l'esperimento della Lezione 7 in scala ridotta:

| | Cosa rimuoviamo | Quante volte |
|---|---|---|
| **Mirato** | il giocatore top-1 per weighted betweenness | 1 |
| **Casuale** | un giocatore di movimento scelto uniformemente a caso (no portiere, no pivot) | $N = 200$ |

Per ciascuna combinazione (squadra × metrica) il danno mirato viene convertito in uno **z-score** all'interno della distribuzione delle rimozioni casuali, e un p-value a una coda riporta la frazione delle realizzazioni casuali che raggiungono o superano il danno mirato.


In [ ]:
metrics = {"avg_path_length": avg_path_length, "efficiency": efficiency}

rows = []
for team in TEAMS:
    G = graphs[team]
    btw = weighted_betweenness(G); pivot = max(btw, key=btw.get)
    pre = {m: f(G) for m, f in metrics.items()}

    # Targeted
    Gt = G.copy(); Gt.remove_node(pivot)
    post = {m: f(Gt) for m, f in metrics.items()}
    for m in metrics:
        rows.append({"team": team, "mode": "targeted", "realisation": 0,
                     "metric": m,
                     "damage": relative_damage(pre[m], post[m], m)})

    # Random null
    rng = np.random.default_rng(RANDOM_SEED)
    outfield = [n for n in G.nodes() if G.nodes[n].get("line") != "GK" and n != pivot]
    for r in range(N_RANDOM):
        n_remove = rng.choice(outfield)
        Gr = G.copy(); Gr.remove_node(n_remove)
        post = {m: f(Gr) for m, f in metrics.items()}
        for m in metrics:
            rows.append({"team": team, "mode": "random", "realisation": r,
                         "metric": m,
                         "damage": relative_damage(pre[m], post[m], m)})

df = pd.DataFrame(rows)

# Summary table.
summary = []
for team in TEAMS:
    for m in metrics:
        sub = df[(df.team == team) & (df.metric == m)]
        t = sub[sub["mode"] == "targeted"]["damage"].iloc[0]
        rvs = sub[sub["mode"] == "random"]["damage"].dropna().values
        rmean, rstd = rvs.mean(), rvs.std(ddof=1)
        z = (t - rmean) / rstd if rstd > 0 else float("nan")
        p = float(np.mean(rvs >= t))
        summary.append({"team": team, "metric": m,
                        "targeted": round(t, 4),
                        "random mean": round(rmean, 4),
                        "random std":  round(rstd, 4),
                        "z": round(z, 2), "p": round(p, 3)})
summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))


In [ ]:
# Hero figure for §3: KPI band + 4×2 grid of histograms with targeted overlay.
_team_it_map = {"Argentina":"Argentina","France":"Francia","Croatia":"Croazia","Morocco":"Marocco"}
fig = plt.figure(figsize=FIG["grid"], facecolor=BG)
gs = fig.add_gridspec(nrows=5, ncols=2, height_ratios=[0.85, 1, 1, 1, 1],
                      left=0.07, right=0.97, top=0.93, bottom=0.05,
                      hspace=0.50, wspace=0.18)

fig.text(0.07, 0.985, "Rimuovere il pivot vs rimuovere un giocatore di movimento a caso",
         ha="left", va="top", fontsize=TEXT["fig_title"], fontweight="semibold",
         color=INK)
fig.text(0.07, 0.957,
         "Per ogni squadra e ogni metrica, il danno della rimozione mirata (linea rossa) è confrontato "
         "con la distribuzione dei danni prodotti da 200 rimozioni casuali di giocatori di movimento.",
         ha="left", va="top", fontsize=TEXT["fig_subtitle"], color=INK_SOFT)

# KPI strip across the top
ax_kpi = fig.add_subplot(gs[0, :])
ranked = summary_df.groupby("team")["z"].mean().sort_values(ascending=False)
most_fragile, least_fragile = ranked.index[0], ranked.index[-1]
max_z = summary_df["z"].max()
mean_z = summary_df["z"].mean()
add_kpi_strip(ax_kpi, [
    (f"{N_RANDOM}",                     "realizzazioni casuali per squadra"),
    (f"{max_z:+.2f}σ",                  f"divario massimo (Francia · Δcammino)"),
    (_team_it_map.get(most_fragile, most_fragile),    "squadra più fragile"),
    (_team_it_map.get(least_fragile, least_fragile),  "squadra più robusta"),
])

# The 4×2 grid
for i, team in enumerate(TEAMS):
    for j, metric in enumerate(["avg_path_length", "efficiency"]):
        ax = fig.add_subplot(gs[i + 1, j])
        sub = df[(df.team == team) & (df.metric == metric)]
        rand_vals = sub[sub["mode"] == "random"]["damage"].dropna().values
        targ_val  = sub[sub["mode"] == "targeted"]["damage"].iloc[0]
        # Histogram of random null
        ax.hist(rand_vals, bins=22, color=TEAM_COLORS[team], alpha=0.55,
                edgecolor="white", linewidth=0.5)
        ax.axvline(0, color=MUTED, lw=1.0, zorder=1)
        # Random mean
        ax.axvline(rand_vals.mean(), color=INK_SOFT, linestyle=":",
                   linewidth=1.4, label="media casuale")
        # Targeted: three-pass glow effect for emphasis
        ax.axvline(targ_val, color=HIGHLIGHT, linewidth=6, alpha=0.18, zorder=2)
        ax.axvline(targ_val, color=HIGHLIGHT, linewidth=2.5, zorder=3,
                   label="mirata")
        # z and p chip in the corner
        z = (targ_val - rand_vals.mean()) / rand_vals.std(ddof=1)
        p_val = float(np.mean(rand_vals >= targ_val))
        sig = "***" if p_val < 0.001 else ("**" if p_val < 0.01 else ("*" if p_val < 0.05 else ""))
        ax.text(0.97, 0.92, f"z = {z:+.2f}  {sig}\np = {p_val:.3f}",
                transform=ax.transAxes, ha="right", va="top",
                fontsize=TEXT["annotation"], family="monospace",
                bbox=dict(boxstyle="round,pad=0.30", fc="white", ec=MUTED,
                          lw=0.8, alpha=0.96))
        # Title (panel-style)
        nice_metric = {"avg_path_length": "Δ lunghezza media cammini",
                       "efficiency": "Δ efficienza"}[metric]
        team_it = {"Argentina":"Argentina","France":"Francia","Croatia":"Croazia","Morocco":"Marocco"}[team]
        style_panel(ax, title=f"{team_it}   ·   {nice_metric}", subtitle=None)
        ax.set_xlabel("danno relativo" if i == 3 else "",
                      fontsize=TEXT["annotation"], color=INK_SOFT)
        ax.set_ylabel("conteggio" if j == 0 else "",
                      fontsize=TEXT["annotation"], color=INK_SOFT)
        if i == 0 and j == 0:
            ax.legend(loc="upper left", frameon=False,
                      fontsize=TEXT["annotation"])

plt.show()


**Cosa abbiamo imparato.**

- Per tutte e quattro le squadre, **il danno della rimozione mirata è superiore alla media casuale** su entrambe le metriche. Il top-1 per betweenness sta facendo qualcosa che i giocatori di movimento casuali non fanno.
- La **Francia** si colloca ben al di fuori della propria distribuzione nulla su entrambe le metriche ($z = +4.84$ su Δcammino; $z = +3.09$ su Δefficienza). Rimuovere Tchouaméni è praticamente impossibile da riprodurre per caso.
- La **Croazia** si colloca *dentro* la massa della propria distribuzione nulla su entrambe le metriche ($z = +0.64$, $z = +1.33$). Rimuovere Modrić sembra una perdita casuale tipica — la squadra ha ridondanza strutturale.
- L'**Argentina** mostra un'interessante **asimmetria** fra le metriche: la Δefficienza mirata è a $z = +3.77$ (molto alta), mentre il Δcammino mirato è a $z = +1.72$ (modesto). La struttura geodetica assorbe la perdita di Otamendi meglio di quanto faccia il flusso globale — un pattern caratteristico dei pivot il cui ruolo è *spazialmente* unico ma localmente ridondante. Ci torneremo nella §5.


---
## 4. Progressione dell'attacco — rimuovere top-1, top-2, top-3

Una singola rimozione ci dice quanto dipende da *un* giocatore. L'esperimento classico della Lezione 7 (Albert, Jeong & Barabási 2000) va oltre: a ogni passo **ricalcoliamo la betweenness sul grafo superstite** e rimuoviamo il nuovo top-1. La curva di danno cumulativo rivela se la squadra possiede hub secondari ridondanti.

Ci fermiamo a tre rimozioni — abbastanza per vedere le curve separarsi, troppo poco per portare il grafo sotto la soglia in cui la misura della lunghezza dei cammini diventa mal definita.


In [ ]:
prog_rows = []
for team in TEAMS:
    G_curr = graphs[team].copy()
    baseline = {m: f(G_curr) for m, f in metrics.items()}
    removed = []
    for step in range(4):
        post = {m: f(G_curr) for m, f in metrics.items()}
        prog_rows.append({
            "team":   team,
            "step":   step,
            "removed so far": ", ".join(short(n) for n in removed) or "—",
            "Δ avg path length": relative_damage(baseline["avg_path_length"],
                                                  post["avg_path_length"],
                                                  "avg_path_length"),
            "Δ efficiency":       relative_damage(baseline["efficiency"],
                                                  post["efficiency"],
                                                  "efficiency"),
        })
        if step < 3:
            btw = weighted_betweenness(G_curr)
            nxt = max(btw, key=btw.get)
            removed.append(nxt)
            G_curr.remove_node(nxt)

prog_df = pd.DataFrame(prog_rows)
print(prog_df.round(3).to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt
from adjustText import adjust_text

# =========================================================
# FIGURE
# =========================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=FIG["wide"],
    facecolor=BG
)

# =========================================================
# MAIN LOOP
# =========================================================

for ax, metric, ylabel, panel_title, panel_subtitle in zip(
    axes,
    ["Δ avg path length", "Δ efficiency"],
    ["Δ lunghezza media cammini (relativa)", "Δ efficienza (relativa)"],
    ["Danno cumulativo alla lunghezza dei cammini",
     "Danno cumulativo sull'efficienza"],
    ["di quanto la palla percorre più strada, in media",
     "quanta parte del flusso di rete viene persa"]
):

    texts = []

    # =====================================================
    # LINES
    # =====================================================

    for team in TEAMS:

        sub = prog_df[prog_df.team == team]
        color = TEAM_COLORS[team]

        # Glow
        ax.plot(
            sub.step,
            sub[metric],
            color=color,
            lw=6,
            alpha=0.18,
            zorder=1
        )

        # Main line
        ax.plot(
            sub.step,
            sub[metric],
            marker="o",
            markersize=9,
            linewidth=2.4,
            color=color,
            label={"Argentina":"Argentina","France":"Francia","Croatia":"Croazia","Morocco":"Marocco"}[team],
            zorder=3
        )

        # =================================================
        # STEP-1 LABEL
        # =================================================

        step1 = sub[sub.step == 1].iloc[0]
        pivot_name = step1["removed so far"]

        t = ax.text(
            1,
            step1[metric],
            f" −{pivot_name}",
            fontsize=TEXT["annotation"] - 0.5,
            color=color,
            fontweight="semibold",
            zorder=5
        )

        texts.append(t)

    # =====================================================
    # AUTO LABEL ADJUST
    # =====================================================

    adjust_text(
        texts,
        ax=ax,
        only_move={"text": "y"},
        arrowprops=dict(
            arrowstyle="-",
            color=INK_SOFT,
            lw=0.6,
            alpha=0.6
        )
    )

    # =====================================================
    # AXES
    # =====================================================

    ax.set_xticks([0, 1, 2, 3])

    ax.set_xticklabels([
        "baseline",
        "+ top-1",
        "+ top-2",
        "+ top-3"
    ])

    ax.set_ylabel(
        ylabel,
        fontsize=TEXT["annotation"],
        color=INK_SOFT
    )

    ax.set_xlabel(
        "rimozioni cumulative (betweenness ricalcolata)",
        fontsize=TEXT["annotation"],
        color=INK_SOFT
    )

    ax.legend(
        loc="upper left",
        frameon=False,
        fontsize=TEXT["annotation"]
    )

    ax.grid(
        True,
        alpha=0.25,
        axis="y"
    )

    # =====================================================
    # CLEAN PANEL TITLES
    # =====================================================

    ax.set_title(
    panel_title,
    fontsize=TEXT["annotation"] + 2,
    color=INK,
    pad=28,
    loc="left",
    fontweight="semibold"
)

    ax.text(
        0,
        1.01,
        panel_subtitle,
        transform=ax.transAxes,
        fontsize=TEXT["annotation"],
        color=INK_SOFT,
        ha="left"
    )

# =========================================================
# GLOBAL TITLE
# =========================================================

fig.suptitle(
    "Progressione dell'attacco — rimozione cumulativa dei top-3 per betweenness ricalcolata",
    fontsize=TEXT["fig_title"],
    fontweight="bold",
    color=INK,
    x=0.06,
    ha="left",
    y=0.98
)

# =========================================================
# GLOBAL SUBTITLE
# =========================================================

fig.text(
    0.06,
    0.92,
    "Le curve divergono nettamente al passo 1 (Francia in testa, Croazia in coda) e convergono parzialmente "
    "al passo 3 — la rete riesce ad assorbire una perdita mirata, non tre.",
    fontsize=TEXT["fig_subtitle"],
    color=INK_SOFT,
    ha="left"
)

# =========================================================
# FINAL LAYOUT
# =========================================================

plt.subplots_adjust(
    left=0.06,
    right=0.98,
    top=0.78,
    bottom=0.12,
    wspace=0.20
)

plt.show()

**Cosa abbiamo imparato.**
- Le curve di danno sono **monotone ma non lineari**: il danno cresce più rapidamente dal baseline al passo 1, poi satura parzialmente man mano che la ridondanza si esaurisce.
- La **Francia** assorbe il colpo più grande al *passo 1* (Tchouaméni), ma al passo 3 (Tchouaméni → Varane → Griezmann) la sua curva è stata superata da Argentina e Croazia.
- La **Croazia** è il caso speculare: il danno minimo al passo 1 di tutto il gruppo, ma il secondo più alto entro il passo 3 — rimuovere Modrić, Gvardiol e Kovačić cumulativamente distrugge il 58% dell'integrità dei cammini. Tre hub quasi intercambiabili attutiscono la perdita di uno, non quella di tutti e tre.
- È il risultato di Albert-Jeong-Barabási in miniatura: le reti eterogenee sono *robuste a poche perdite, fragili a un attacco coordinato sugli hub*.


---
## 5. Il predittore di dominanza del pivot — la RQ1 prevede la RQ3?

Abbiamo ora due misure indipendenti per squadra:

- **Dominanza dalla RQ1:** rapporto top-1 / top-2 della weighted betweenness, calcolato nella §1.
- **Fragilità dalla RQ3:** lo z-score medio sulle due metriche di danno della §3.

Se la nostra lettura è giusta, le due misure devono essere **correlate positivamente**. Più una squadra ha un pivot dominante *strutturalmente*, peggio dovrebbe comportarsi *dinamicamente* quando quel pivot viene rimosso.

Con $n = 4$ questa non è un'affermazione statistica — è un **controllo di consistenza fra due esperimenti sullo stesso dataset**.


In [ ]:
# Build the joint table.
fragility = summary_df.groupby("team")["z"].mean().rename("fragility z").to_frame()
dom = pd.DataFrame({"team": TEAMS,
                    "top1/top2": setup_df.set_index("team").loc[TEAMS, "top1/top2"].values
                   }).set_index("team")
joint = dom.join(fragility)
print(joint.round(2).to_string())

r,  _ = pearsonr(joint["top1/top2"], joint["fragility z"])
rs, _ = spearmanr(joint["top1/top2"], joint["fragility z"])
print(f"\nPearson  r = {r:+.2f}")
print(f"Spearman ρ = {rs:+.2f}   (n = 4 teams)")


In [ ]:
# Flagship composite: scatter + four pitch mini-cards on the side.
# Style template: the "publication-ready" composite from notebook 04 §D.

fig = plt.figure(figsize=FIG["flagship"], facecolor=BG)
gs = fig.add_gridspec(
    nrows=4, ncols=12,
    left=0.05, right=0.97, top=0.86, bottom=0.10,
    hspace=1.0, wspace=0.6,
)

# --- Banner -----------------------------------------------------------------
fig.text(0.05, 0.965,
         "La dominanza del pivot predice la fragilità della rete",
         ha="left", va="top", fontsize=TEXT["fig_title"] + 2,
         fontweight="semibold", color=INK)
fig.text(0.05, 0.928,
         "Due esperimenti sulle stesse quattro squadre convergono sullo stesso ordine. "
         "Francia — pivot più dominante, rete più fragile. Croazia — pivot più "
         "intercambiabili, rete più robusta.",
         ha="left", va="top", fontsize=TEXT["fig_subtitle"], color=INK_SOFT,
         wrap=True)

# --- Main panel: scatter ----------------------------------------------------
ax_main = fig.add_subplot(gs[0:4, 0:7])
# Best-fit line under data
slope, intercept = np.polyfit(joint["top1/top2"], joint["fragility z"], 1)
xs = np.linspace(joint["top1/top2"].min() * 0.92,
                 joint["top1/top2"].max() * 1.08, 50)
ax_main.plot(xs, slope * xs + intercept, "--", color=INK_SOFT, lw=1.4,
             alpha=0.6, zorder=1)
# Glow then sharp point per team
for team in TEAMS:
    x = joint.loc[team, "top1/top2"]
    y = joint.loc[team, "fragility z"]
    color = TEAM_COLORS[team]
    ax_main.scatter(x, y, s=2100, color=color, ec="white", lw=2.5,
                    alpha=0.18, zorder=2)
    ax_main.scatter(x, y, s=600, color=color, ec="white", lw=2.5, zorder=3)
    offsets = {"Argentina": (16, 14), "France": (-22, -22),
               "Croatia": (16, -14), "Morocco": (-12, 18)}
    ox, oy = offsets[team]
    team_it = {"Argentina":"Argentina","France":"Francia","Croatia":"Croazia","Morocco":"Marocco"}[team]
    ax_main.annotate(team_it, (x, y), xytext=(ox, oy), textcoords="offset points",
                     fontsize=13, fontweight="semibold", color=color)

# Correlation annotation in the corner
ax_main.text(0.97, 0.05,
             f"Pearson  r = {r:+.2f}\nSpearman ρ = {rs:+.2f}\n(n = 4)",
             transform=ax_main.transAxes, ha="right", va="bottom",
             fontsize=TEXT["annotation"], family="monospace",
             bbox=dict(boxstyle="round,pad=0.35", fc="white",
                       ec=MUTED, lw=0.8, alpha=0.96))

# Title for the panel: explicit, doesn't use style_panel (which collides with banner)
ax_main.set_title("Dominanza dalla RQ1   vs   fragilità dalla RQ3",
                  loc="left", pad=10,
                  fontsize=TEXT["panel_title"], fontweight="semibold", color=INK)
ax_main.set_xlabel("rapporto top-1 / top-2 della betweenness  (RQ1)",
                   fontsize=TEXT["annotation"] + 0.5, color=INK_SOFT)
ax_main.set_ylabel("z-score di fragilità  (RQ3)",
                   fontsize=TEXT["annotation"] + 0.5, color=INK_SOFT)
ax_main.grid(True, alpha=0.25)
ax_main.set_xlim(0.9, 3.45)
ax_main.set_ylim(0.4, 4.5)

# --- Side: four pitch mini-cards with the pivot highlighted -----------------
for i, team in enumerate(TEAMS):
    ax = fig.add_subplot(gs[i, 7:])
    G = graphs[team]
    btw = weighted_betweenness(G)
    pivot = max(btw, key=btw.get)
    color = TEAM_COLORS[team]
    # Pitch
    draw_pitch(ax, color=PITCH_BG, line=PITCH_LN, lw=0.6)
    # Edges in faint gold (compute wmax once outside the loop)
    wmax = max(d["weight"] for _, _, d in G.edges(data=True))
    for u, v, d in G.edges(data=True):
        if d["weight"] < 6: continue
        if "x" not in G.nodes[u] or "x" not in G.nodes[v]: continue
        x1, y1 = G.nodes[u]["x"], G.nodes[u]["y"]
        x2, y2 = G.nodes[v]["x"], G.nodes[v]["y"]
        ax.plot([x1, x2], [y1, y2], color="#bba266",
                alpha=0.18 + 0.6*(d["weight"]/wmax),
                lw=0.35 + 2.5*(d["weight"]/wmax),
                zorder=2)
    # Nodes — pivot in ACCENT (orange) to be distinguishable on any team colour
    btw_max = max(btw.values())
    for n in G.nodes():
        if "x" not in G.nodes[n]: continue
        x, y = G.nodes[n]["x"], G.nodes[n]["y"]
        size = 30 + 350 * btw.get(n, 0) / btw_max
        is_pivot = (n == pivot)
        node_color = ACCENT if is_pivot else color
        ec = "white"
        lw_e = 2.5 if is_pivot else 1.0
        zorder_n = 5 if is_pivot else 3
        # Glow under pivot
        if is_pivot:
            ax.scatter(x, y, s=size*2.4, color=ACCENT, alpha=0.30, zorder=4)
        ax.scatter(x, y, s=size, color=node_color, ec=ec, lw=lw_e,
                   zorder=zorder_n)
    # Title in white on the dark pitch
    team_it = {"Argentina":"Argentina","France":"Francia","Croatia":"Croazia","Morocco":"Marocco"}[team]
    ax.text(60, 75, f"{team_it}  ·  pivot = {short(pivot)}",
            ha="center", va="top",
            fontsize=TEXT["annotation"] + 0.5, color="white",
            fontweight="semibold",
            bbox=dict(boxstyle="round,pad=0.3", fc="#0d1f17", ec="none",
                      alpha=0.6))
    # Stats below
    z_score = float(fragility.loc[team, "fragility z"])
    dom_score = float(dom.loc[team, "top1/top2"])
    text_below_axes(
        ax,
        f"top1/top2 = {dom_score:.2f}     fragilità z = {z_score:+.2f}",
        y=-0.10, mono=True, box=True,
    )

# Caption line tying everything together
fig.text(0.05, 0.035,
         "I marker più grandi e luminosi identificano il pivot strutturale di ogni squadra. "
         "L'Argentina si trova sopra la linea di regressione perché il ruolo di broker di Otamendi è "
         "spazialmente unico dentro la squadra — una raffinatura segnalata nel report.",
         ha="left", va="top", fontsize=TEXT["annotation"], color=INK_SOFT,
         style="italic", wrap=True)

plt.show()


**Cosa abbiamo imparato.**
- **Pearson $r = +0.90$, Spearman $\rho = +0.80$**. Due esperimenti indipendenti su quattro squadre producono lo stesso ordinamento di fragilità: **Francia ▸ Argentina ▸ Marocco ▸ Croazia**.
- La retta è un fit stretto su tre punti. Francia (Tchouaméni) e Croazia (centrocampisti intercambiabili) stanno agli estremi. Il Marocco si colloca vicino alla Croazia, coerentemente col fatto che Hakimi e Amrabat sono quasi pari per betweenness.
- L'**Argentina è l'outlier sopra la linea**. Il suo z-score è più alto di quanto il rapporto top-1 / top-2 prevedrebbe. La lettura della §3 — che la rimozione di Otamendi fa più male all'efficienza che alla lunghezza dei cammini — si generalizza qui: l'*unicità spaziale* del ruolo di un pivot non viene catturata da un punteggio monodimensionale come il rapporto fra centralità. Una misura di dominanza più raffinata (pesata sulla separazione spaziale fra i top pivot) riporterebbe probabilmente l'Argentina sulla linea.


---
## In sintesi

- La **fragilità della rete dei passaggi** di una squadra si può leggere direttamente dalla struttura: è ben predetta dal divario fra top-1 e top-2 della weighted betweenness (**$r = +0.90$, $n = 4$**).
- La metrica $S(q)$ della NS07 non si trasferisce alle reti dense pesate; due sostitute — Δ lunghezza media dei cammini e Δ efficienza — ne preservano lo spirito su reti troppo dense per frammentarsi con la rimozione di un singolo nodo.
- Il risultato di Albert-Jeong-Barabási sulle reti scale-free vale *qualitativamente* anche su questi piccoli grafi pesati del calcio: il danno da attacco mirato cresce più rapidamente del danno da fallimento casuale, e il divario è più ampio quando un hub è strutturalmente dominante.
- **Lettura tattica.** La Francia di Tchouaméni è stata la squadra la cui struttura di passaggi dipendeva più precariamente da un singolo giocatore. I tre centrocampisti quasi intercambiabili della Croazia (Modrić, Brozović, Kovačić) erano l'opposto strutturale — un nucleo ridondante. L'Argentina ha vinto il torneo con una struttura più vicina a quella della Francia che a quella della Croazia, e questa è la tensione irrisolta che già la RQ1 fa emergere.
- **Filo aperto.** La posizione di outlier dell'Argentina indica una raffinatura: un punteggio di dominanza *spazialmente consapevole* peserebbe l'unicità geometrica del pivot sul campo, non solo il divario di rango nella betweenness. Inserita nella sezione di future work del progetto.

---

### Cinque domande a cui un* student* dovrebbe saper rispondere dopo questo notebook

1. Perché $S(q)$ della NS07 non funziona su queste reti, e cosa la sostituisce?
2. Cosa ci dice sul ruolo di un pivot uno z-score alto sulla Δ-efficienza ma basso sul Δ-cammino?
3. Perché le curve di attacco mirato delle quattro squadre convergono al passo 3?
4. Con $n = 4$, perché $r = +0.90$ *non* è una scoperta statistica, e cos'è invece?
5. Perché l'Argentina si trova sopra la linea predittiva, e cosa cambierebbe se la misura di dominanza fosse spaziale?
